# DeepSage — Inference Server Test

Before running this notebook:
```bash
deepsage pick          # download & register a model
deepsage switch <name> # set it as active
deepsage serve         # start the server on http://127.0.0.1:8888
```

In [1]:
# Install dependencies if needed
# %pip install openai requests sseclient-py

In [2]:
import requests

BASE_URL = "http://127.0.0.1:8888"

# ── Config ────────────────────────────────────────────────────────────────────
# Change port here if you started the server with --port XXXX
# e.g. deepsage serve --port 9000  →  BASE_URL = "http://127.0.0.1:9000"

## 1 · Health Check

In [3]:
resp = requests.get(f"{BASE_URL}/health", timeout=5)
print(resp.status_code, resp.json())

200 {'active_model': 'Llama-3.2-1B-Instruct', 'status': 'ok'}


## 2 · List Registered Models

In [4]:
resp = requests.get(f"{BASE_URL}/v1/models")
models = resp.json()["data"]
for m in models:
    print(f"  {m['id']:30s}  backend={m['owned_by']}")

# Use the active model reported by /health
health = requests.get(f"{BASE_URL}/health").json()
active_name = health.get("active_model", "")
MODEL = active_name if active_name else (models[0]["id"] if models else "unknown")
print(f"\nActive model: {MODEL}")


  DeepSeek-R1-0528-Qwen3-8B       backend=ollama
  DeepSeek-R1-7B                  backend=ollama
  Llama-3.2-1B-Instruct           backend=llamacpp

Active model: Llama-3.2-1B-Instruct


## 3 · Simple Chat Completion (requests)

In [5]:
# Debug: raw Ollama response
import requests, json as _json
raw = requests.post(f"{BASE_URL}/v1/chat/completions", json={
    "model": MODEL,
    "messages": [{"role": "user", "content": "Hi"}],
    "stream": False
})
print("Status:", raw.status_code)
print(_json.dumps(raw.json(), indent=2))

Status: 200
{
  "id": "chatcmpl-2a0b543b3f36407d8fb3e83d44804494",
  "object": "chat.completion",
  "created": 1776755105,
  "model": "Llama-3.2-1B-Instruct",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "How can I assist you today?"
      },
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "prompt_tokens": 0,
    "completion_tokens": 0,
    "total_tokens": 0
  }
}


In [6]:
payload = {
    "model": MODEL,
    "messages": [
        {"role": "user", "content": "What is the capital of France? Answer in one sentence."}
    ],
    "stream": False
}

resp = requests.post(f"{BASE_URL}/v1/chat/completions", json=payload)
data = resp.json()
print(data["choices"][0]["message"]["content"])

The capital of France is Paris.


In [7]:
data


{'id': 'chatcmpl-c2e9b3c09ff549e3bf6e0b4feec984c3',
 'object': 'chat.completion',
 'created': 1776755108,
 'model': 'Llama-3.2-1B-Instruct',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'The capital of France is Paris.'},
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0}}

## 4 · Chat Completion (openai SDK)

In [8]:
from openai import OpenAI

client = OpenAI(
    base_url=f"{BASE_URL}/v1",
    api_key="none"   # DeepSage does not require an API key
)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system",  "content": "You are a concise assistant."},
        {"role": "user",    "content": "Explain what a large language model is in two sentences."}
    ]
)

print(response.choices[0].message.content)

A large language model is a computer system that uses artificial intelligence to generate human-like text, speech, or other forms of language, by analyzing and predicting the patterns of language data. These models, often referred to as transformer models, are trained on vast amounts of text data and can produce coherent and context-specific responses, enabling applications such as chatbots, virtual assistants, and language translation systems.


## 5 · Streaming Response

In [9]:
import json

payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "Count from 1 to 10, one number per line."}],
    "stream": True
}

with requests.post(f"{BASE_URL}/v1/chat/completions", json=payload, stream=True) as resp:
    for line in resp.iter_lines():
        if not line:
            continue
        text = line.decode("utf-8")
        if text.startswith("data: "):
            data = text[6:]
            if data == "[DONE]":
                break
            chunk = json.loads(data)
            delta = chunk["choices"][0]["delta"].get("content", "")
            print(delta, end="", flush=True)

print()  # newline after stream

None1

2

3

4

5

6

7

8

9

10


## 6 · Streaming with openai SDK

In [10]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Write a haiku about machine learning."}],
    stream=True
)

for chunk in stream:
    delta = chunk.choices[0].delta.content or ""
    print(delta, end="", flush=True)

print()

Code and data dance
Artificial minds awake
Learning's gentle grasp


## 7 · Multi-turn Conversation

In [11]:
history = [{"role": "system", "content": "You are a helpful assistant."}]

def chat(user_message: str) -> str:
    history.append({"role": "user", "content": user_message})
    resp = client.chat.completions.create(model=MODEL, messages=history)
    reply = resp.choices[0].message.content
    history.append({"role": "assistant", "content": reply})
    return reply

print(chat("My name is Alice. Remember it."))
print(chat("What is my name?"))

I'll make sure to remember that you're Alice. Is there anything else I can help you with today, Alice?
I remember now, your name is Alice.


## 8 · Legacy Completions Endpoint

In [12]:
payload = {
    "model": MODEL,
    "prompt": "The three laws of robotics are:",
    "stream": False
}

resp = requests.post(f"{BASE_URL}/v1/completions", json=payload)
print(resp.json()["choices"][0]["message"]["content"])

A classic reference. The three laws of robotics, as first proposed by R. Daneel Olivaw in the 2004 movie "I, Robot," are:

1. A robot may not injure a human being or, through inaction, allow a human being to come to harm.
2. A robot must obey the orders of human beings except where such orders would conflict with the First Law.
3. A robot must protect its own existence as long as such protection does not conflict with the First or Second Law.

These laws are often seen as a framework for understanding the moral and ethical implications of creating and interacting with robots, and have since been referenced in various forms of media, including literature, film, and video games.


## 9 · Batch Inference

In [13]:
import concurrent.futures

prompts = [
    "What is 2 + 2?",
    "Name one planet in the solar system.",
    "What colour is the sky?",
]

def infer(prompt: str) -> str:
    r = requests.post(f"{BASE_URL}/v1/chat/completions", json={
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False
    })
    return r.json()["choices"][0]["message"]["content"].strip()

with concurrent.futures.ThreadPoolExecutor(max_workers=3) as pool:
    results = list(pool.map(infer, prompts))

for prompt, result in zip(prompts, results):
    print(f"Q: {prompt}")
    print(f"A: {result}\n")

Q: What is 2 + 2?
A: 2 + 2 = 4.

Q: Name one planet in the solar system.
A: Mars is one of the planets in our solar system.

Q: What colour is the sky?
A: The color of the sky can vary depending on the time of day, atmospheric conditions, and other factors. In general, the sky appears blue to our eyes when it's a clear, sunny day with little to no atmospheric particles like dust, smoke, or pollution.

However, when the sun is at its peak in the sky, the color of the sky can shift to different hues, such as:

* Yellow or orange during sunrise and sunset, when the light has to travel through more of the Earth's atmosphere to reach our eyes.
* Pink or reddish during sunrise and sunset, when the light is scattered more in a particular direction.

On cloudy days, the sky can appear gray or white due to the presence of atmospheric particles.

It's worth noting that the color of the sky can also be influenced by other factors, such as pollution, dust, and water vapor in the air.

What time of

## 10 · Switch Model at Runtime

In [14]:
import subprocess

# List registered models from the CLI
result = subprocess.run(["deepsage", "list"], capture_output=True, text=True)
print(result.stdout)

# Switch to a different model (restart the server after switching)
# subprocess.run(["deepsage", "switch", "other-model-name"])

 Registered Models
────────────────────────────────────────────────────────────────────────
Name                   Backend    VRAM       Active
────────────────────────────────────────────────────────────────────────
DeepSeek-R1-0528-Qwen3-8B ollama     5.5G       ○
DeepSeek-R1-7B         ollama     auto       ○
Llama-3.2-1B-Instruct  llamacpp   auto       ● active

 Inference Server
────────────────────────────────────────────────────────────────────────
  ● Running  model: Llama-3.2-1B-Instruct
  Endpoint : http://127.0.0.1:8888/v1/chat/completions

  Python:
    from openai import OpenAI
    client = OpenAI(base_url="http://127.0.0.1:8888/v1", api_key="none")
    r = client.chat.completions.create(model="Llama-3.2-1B-Instruct", messages=[{"role":"user","content":"Hello"}])
    print(r.choices[0].message.content)

